In [0]:
from pyspark.sql import functions as F

trips = spark.table("workspace.bronze.trips_raw")

# check how many rows we're about to drop, before we drop them
display(spark.sql("""
    SELECT 
        SUM(CASE WHEN trip_duration = 0 THEN 1 ELSE 0 END) AS zero_duration,
        SUM(CASE WHEN trip_duration > 86400 THEN 1 ELSE 0 END) AS over_24h,
        COUNT(*) AS total_rows
    FROM workspace.bronze.trips_raw
"""))

zero_duration,over_24h,total_rows
5479,39,7812520


In [0]:
trips_filtered = trips.filter(
    (F.col("trip_duration") > 0) & (F.col("trip_duration") <= 86400)
)

print(f"Before: {trips.count():,}")
print(f"After:  {trips_filtered.count():,}")

Before: 7,812,520
After:  7,807,002


In [0]:
trips_clean = (trips_filtered
    .withColumn("start_station_name", F.trim(F.initcap(F.col("start_station_name"))))
    .withColumn("end_station_name", F.trim(F.initcap(F.col("end_station_name"))))
)

In [0]:
display(spark.sql("DESCRIBE workspace.bronze.trips_raw"))

col_name,data_type,comment
trip_id,int,null
trip_duration,int,null
start_station_id,int,null
start_time,timestamp,null
start_station_name,string,null
end_station_id,int,null
end_time,timestamp,null
end_station_name,string,null
bike_id,int,null
user_type,string,null


In [0]:
trips_with_date = trips_clean.withColumn("trip_date", F.to_date(F.col("start_time")))

weather = spark.table("workspace.bronze.weather_raw").drop("_load_ts")

trips_joined = trips_with_date.join(
    weather,
    trips_with_date.trip_date == weather.time,
    how="left"
)

In [0]:
trips_joined.createOrReplaceTempView("trips_joined_check")

display(spark.sql("""
    SELECT COUNT(*) AS total_rows, 
           SUM(CASE WHEN temperature_2m_mean IS NULL THEN 1 ELSE 0 END) AS missing_weather
    FROM trips_joined_check
"""))

total_rows,missing_weather
7807002,0


In [0]:
# drop duplicate 'time' column from weather since we already have trip_date
trips_final = trips_joined.drop("time")

trips_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.trips_clean")

In [0]:
display(spark.sql("SELECT * FROM workspace.silver.trips_clean LIMIT 5"))
display(spark.sql("SELECT COUNT(*) FROM workspace.silver.trips_clean"))

trip_id,trip_duration,start_station_id,start_time,start_station_name,end_station_id,end_time,end_station_name,bike_id,user_type,bike_model,_load_ts,_source_file,trip_date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,windspeed_10m_max
34637659,193,7192,2025-01-01T00:02:15.000Z,Harbord St / Clinton St,7155,2025-01-01T00:05:28.000Z,Harbord St / Clinton St,4422,Member,ICONIC,2026-10-09T19:16:34.624Z,dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0
34637678,464,7508,2025-01-01T00:14:07.000Z,Berkeley St / Dundas St E - Smart,7042,2025-01-01T00:21:51.000Z,Berkeley St / Dundas St E - Smart,4618,Member,ICONIC,2026-10-09T19:16:34.624Z,dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0
34637663,770,7094,2025-01-01T00:07:21.000Z,14 Arundel Ave,7777,2025-01-01T00:20:11.000Z,14 Arundel Ave,7975,Member,EFIT G5,2026-10-09T19:16:34.624Z,dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0
34637679,559,7032,2025-01-01T00:14:09.000Z,Augusta Ave / Dundas St W,7541,2025-01-01T00:23:28.000Z,Augusta Ave / Dundas St W,3179,Member,ICONIC,2026-10-09T19:16:34.624Z,dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0
34637658,116,7017,2025-01-01T00:02:13.000Z,Widmer St / Adelaide St W - Smart,7102,2025-01-01T00:04:09.000Z,Widmer St / Adelaide St W - Smart,3177,Member,ICONIC,2026-10-09T19:16:34.624Z,dbfs:/Volumes/workspace/bronze/raw_files/bikeshare_2025_01.csv,2025-01-01,3.7,0.2,1.9,2.0,2.0,23.0


COUNT(*)
7807002
